human lung cancer

导入相关包

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [2]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/human_lung_cancer/SMI_Lung.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_Geneformer'
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/geneformer_human_lung_cancer_emb.parquet"


读取嵌入

In [3]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [4]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial', 'X_Geneformer'

In [5]:
adata.obsm[cell_emb_col]

array([[-0.53651196,  0.21571891,  0.19943973, ..., -1.8232745 ,
        -1.2567023 ,  0.36680475],
       [-0.51719505,  0.2110504 , -0.05354029, ..., -2.0593414 ,
        -1.327544  ,  0.1473998 ],
       [-0.44783267,  0.34775352, -0.12345362, ..., -1.9827502 ,
        -1.207774  ,  0.39436337],
       ...,
       [-1.0724655 , -0.514914  , -1.2409027 , ..., -2.5596852 ,
        -0.9047971 ,  0.8700316 ],
       [-0.77731943, -0.5370814 , -1.2282981 , ..., -2.7611084 ,
        -1.2142248 ,  0.318782  ],
       [-1.5676407 , -0.42285317, -1.1035324 , ..., -2.5346692 ,
        -0.97355884,  0.16322674]], dtype=float32)

In [6]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    uns: 'neighbors'
    obsm: 'spatial', 'X_Geneformer'
    obsp: 'distances', 'connectivities'

In [7]:
max_value = 0
metrics = {"method": "Geneformer", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
save_label_df = None

In [8]:

for used_res in res_list:
    sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
    true_labels = np.array(adata.obs[celltype_col])
    cluster_labels = np.array(adata.obs[key_added])
    FMI = fowlkes_mallows_score(true_labels, cluster_labels)
    homogeneity = homogeneity_score(true_labels, cluster_labels)
    v_measure = v_measure_score(true_labels, cluster_labels)
    ami = adjusted_mutual_info_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(true_labels, cluster_labels)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

    n_cluster = len(adata.obs[key_added].unique())
    n_celltype = len(adata.obs[celltype_col].unique())
    if mean_value > max_value:
        save_label_df = adata.obs[key_added]
        metrics["ARI"] = ari
        metrics["NMI"] = nmi
        metrics["AMI"] = ami
        metrics["HS"] = homogeneity
        metrics["VM"] = v_measure
        metrics["FMI"] = FMI
        metrics["mean value"] = mean_value
        max_value = mean_value

    print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

/tmp/ipykernel_2628354/737231793.py:2: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)


resolution = 0.1 | n_celltype = 18 | n_cluster = 2 | mean_value = 0.109 | ARI = 0.075 |  NMI = 0.046 | AMI = 0.046 | HS = 0.031 | VM = 0.046 | FMI = 0.408 
resolution = 0.2 | n_celltype = 18 | n_cluster = 3 | mean_value = 0.088 | ARI = 0.013 |  NMI = 0.052 | AMI = 0.052 | HS = 0.039 | VM = 0.052 | FMI = 0.32 
resolution = 0.3 | n_celltype = 18 | n_cluster = 3 | mean_value = 0.081 | ARI = 0.015 |  NMI = 0.046 | AMI = 0.046 | HS = 0.035 | VM = 0.046 | FMI = 0.299 
resolution = 0.4 | n_celltype = 18 | n_cluster = 4 | mean_value = 0.067 | ARI = -0.026 |  NMI = 0.047 | AMI = 0.047 | HS = 0.039 | VM = 0.047 | FMI = 0.246 
resolution = 0.5 | n_celltype = 18 | n_cluster = 4 | mean_value = 0.068 | ARI = -0.021 |  NMI = 0.046 | AMI = 0.046 | HS = 0.038 | VM = 0.046 | FMI = 0.25 
resolution = 0.6 | n_celltype = 18 | n_cluster = 6 | mean_value = 0.07 | ARI = 0.004 |  NMI = 0.051 | AMI = 0.051 | HS = 0.048 | VM = 0.051 | FMI = 0.213 
resolution = 0.7 | n_celltype = 18 | n_cluster = 6 | mean_value =

In [9]:
save_label_df

unique_ID
1-2        1
1-3        1
1-4        1
1-6        1
1-7        1
          ..
20-4048    0
20-4049    0
20-4050    0
20-4051    0
20-4052    0
Name: leiden, Length: 87589, dtype: category
Categories (2, object): ['0', '1']

In [10]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/lung_cancer/geneformer_human_lung_cancer_labels_repeat_{repeat_times}.csv"

In [11]:
save_label_df.to_csv(save_label_df_path)

In [12]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [13]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_repeat_{repeat_times}.csv"

In [14]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))